# heartbeat

> what wakes the tick on this machine

In [ ]:
#| default_exp heartbeat

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

There is no scheduler daemon here, and there is not going to be one. `Pob.tick` is a function,
and something outside this process has to call it every minute. Every desktop operating system
already runs a program whose whole job is that. This module installs one entry in it.

Two clocks, and only one of them is smart. The machine's clock knocks once a minute and knows
nothing. pobblebonk's clock lives in SQLite, knows about `cron='0 17 * * *'`, `pause`, `retries`
and `catchup`, and only advances when somebody calls `tick`. So the entry we install says
*every minute, forever*, and never anything else. Mirroring a schedule into the machine's clock
would put the same instruction in two places, and the copy in the machine has no idea what
`needs='shopping'` means.

Three machines, three schedulers. `CronBeat` writes a crontab line. `LaunchdBeat` writes a
LaunchAgent, which is what macOS actually wants. `TaskBeat` calls `schtasks`. They share the
`Beat` interface, and `beat()` picks.

In [ ]:
#| export
import os, plistlib, subprocess, sys
import xml.etree.ElementTree as ET
from pathlib import Path

from fastcore.all import first, ifnone

One tag names everything: the launcher file, its log, and the job in whichever scheduler.
Change it and you have a second, independent heartbeat, which is how one machine ticks two
databases.

Sixty seconds is not a preference. cron cannot go below a minute and neither can
`schtasks /sc minute`. launchd would happily take thirty, but a heartbeat that only two of the
three can keep is not one heartbeat.

In [ ]:
#| export
BEAT = 'pobblebonk'                    # names the launcher, its log, and the scheduled job
EVERY = 60                             # seconds between beats; the floor all three schedulers share
HOME = Path.home()/'.pobblebonk'       # where the launcher and its log go

## The launcher

Every backend schedules the same thing: one small file. It would be possible to hand each
scheduler the full command instead, and it is a bad idea. cron, launchd and `schtasks` disagree
about quoting; `schtasks /tr` truncates long commands; and none of the three tells you why the
thing you scheduled did nothing.

A file solves all three. The schedulers only ever see a path with no arguments and nothing to
quote, and when it misbehaves you can run it yourself and watch.

`cmd` goes in unescaped, because it *is* a command line — `uv run --directory ~/app python -m
app.tick` has to reach the shell as those five words. The redirect is `>>`, so the log holds the
last few ticks rather than the last one. `newline=''` is there because Python would otherwise
translate the `\r\n` of the Windows file into `\r\r\n`, which `cmd` reads as a blank line between
every line.

In [ ]:
#| export
def launcher(cmd:str,           # the command one beat runs, as you would type it
             dirn=None,         # where to write it; `~/.pobblebonk` by default
             tag:str=BEAT,      # names this file and its log
             win:bool=None      # write a Windows `.cmd`; this platform decides by default
) -> Path:
    'Write the one file every backend schedules, and give back its path.'
    win = sys.platform == 'win32' if win is None else win
    d = Path(ifnone(dirn, HOME)).expanduser()
    d.mkdir(parents=True, exist_ok=True)
    log, p = d/f'{tag}.log', d/f'{tag}.{"cmd" if win else "sh"}'
    p.write_text(f'@echo off\r\n{cmd} >> "{log}" 2>&1\r\n' if win else
                 f'#!/bin/sh\nexec {cmd} >> "{log}" 2>&1\n', newline='')
    if not win: p.chmod(0o755)
    return p

The point of writing a file rather than a command string is that the file runs. So run it:

In [ ]:
import subprocess
from tempfile import mkdtemp

d = Path(mkdtemp())
sh = launcher('/bin/echo ticked', d, tag='demo')

test_eq(sh.name, 'demo.sh')
test_eq(subprocess.run([str(sh)]).returncode, 0)
test_eq((d/'demo.log').read_text(), 'ticked\n')

In [ ]:
# A second beat appends. A log holding only the most recent tick would be useless the one
# morning you go looking for what happened at 3am.
subprocess.run([str(sh)])
test_eq((d/'demo.log').read_text().count('ticked'), 2)

In [ ]:
# The same command, told it is on Windows: a different file, and a shell that has never heard
# of `#!`.
cmd = launcher('py -m app.tick', d, tag='demo', win=True)
test_eq(cmd.name, 'demo.cmd')
test_eq(cmd.read_text().splitlines()[0], '@echo off')

## What a scheduler has to do

Three methods. `install` **replaces** what the same tag installed before rather than adding to
it, because a heartbeat installed twice ticks twice, and two ticks in the same minute means two
processes racing for the same fires. Each backend gets that property from something the OS
already offers: a comment on a crontab line, a plist filename, `schtasks /f`.

In [ ]:
#| export
class Beat:
    """One machine's way of running a file every minute.

    `install` replaces what `tag` installed before. `installed` reads back what is really
    scheduled, not what we hoped, so it is the honest way to answer "is this thing on".
    """
    def install(self, script, tag:str=BEAT):
        'Schedule `script` every minute under `tag`, replacing any earlier one.'
        raise NotImplementedError
    def installed(self, tag:str=BEAT) -> str:
        'The script `tag` runs now, or None.'
        raise NotImplementedError
    def uninstall(self, tag:str=BEAT) -> bool:
        'Remove it. False if there was nothing to remove.'
        raise NotImplementedError

## cron

The crontab is one file per user, holding your other jobs too, so the dangerous operation is not
adding a line but rewriting the file around it. That is the whole reason
[python-crontab](https://pypi.org/project/python-crontab/) is a dependency: it parses the file,
finds our line by its trailing comment, and leaves everything else alone. It is an optional
extra, `pobblebonk[cron]`, since a Mac does not need it.

`tabfile` exists so this can be pointed at a crontab that is not yours. The tests below use it.

In [ ]:
#| export
CRON_LINE = '* * * * *'                # every minute: cron's finest resolution, and enough

class CronBeat(Beat):
    'A line in the user crontab. Linux, and anywhere without something better.'
    def __init__(self,
                 tabfile=None):        # a crontab file to edit; None is the live user crontab
        self.tabfile = tabfile

    def _tab(self):
        try: from crontab import CronTab
        except ImportError: raise ImportError("a crontab needs `uv add 'pobblebonk[cron]'`")
        return CronTab(user=True) if self.tabfile is None else CronTab(tabfile=str(self.tabfile))

    def install(self, script, tag:str=BEAT):
        tab = self._tab()
        tab.remove_all(comment=str(tag))
        tab.new(command=str(script), comment=str(tag)).setall(CRON_LINE)
        tab.write()
        return script

    def installed(self, tag:str=BEAT) -> str:
        job = first(self._tab().find_comment(str(tag)))
        return str(job.command) if job else None

    def uninstall(self, tag:str=BEAT) -> bool:
        tab = self._tab()
        if not tab.remove_all(comment=str(tag)): return False
        tab.write()
        return True

A crontab with somebody's nightly backup already in it. Everything below is about that line
surviving.

In [ ]:
tabfile = d/'crontab'
tabfile.write_text('# nightly backup\n0 3 * * * /usr/bin/backup\n')

cron = CronBeat(tabfile)
cron.install('/home/k/.pobblebonk/pobblebonk.sh')

print(tabfile.read_text())

In [ ]:
test_eq(cron.installed(), '/home/k/.pobblebonk/pobblebonk.sh')
assert '* * * * * /home/k/.pobblebonk/pobblebonk.sh' in tabfile.read_text()
assert '0 3 * * * /usr/bin/backup' in tabfile.read_text()

In [ ]:
# Installing again replaces the line rather than adding a second one.
cron.install('/home/k/.pobblebonk/pobblebonk.sh')
test_eq(tabfile.read_text().count('pobblebonk.sh'), 1)

In [ ]:
# Removing ours removes only ours, and says whether there was anything there.
test_eq(cron.uninstall(), True)
test_eq(cron.installed(), None)
test_eq(cron.uninstall(), False)
assert '0 3 * * * /usr/bin/backup' in tabfile.read_text()

## launchd

macOS has cron, and you should not use it. A cron job there runs without Full Disk Access, so a
callback that reads `~/Documents` fails with an error that does not mention permissions. And cron
loses any slot the machine slept through, whereas launchd fires a missed `StartInterval` once on
waking — which is exactly the knock `catchup` needs to make up the fires from a shut laptop.

A LaunchAgent is a plist in `~/Library/LaunchAgents`, and the filename is the identity, so
replacing is just writing over it. `launchctl` still has to be told, and the order matters:
`bootout` an old job before `bootstrap`ping the new one, or launchd keeps running the version it
already loaded.

`_run` is how this module talks to a scheduler binary. `check` turns a complaint into an
exception, for the calls where silence would mean a heartbeat that was never installed. A missing
binary is the same kind of failure as a refusal, so it arrives the same way. Output is decoded
here rather than by `subprocess`, because `schtasks` answers in UTF-16 and the locale guess
`text=True` makes would turn it into rubble.

In [ ]:
#| export
def _text(b:bytes) -> str:
    'Decode scheduler output, which `schtasks` hands back as UTF-16.'
    return b.decode('utf-16') if b[:2] in (b'\xff\xfe', b'\xfe\xff') else b.decode(errors='replace')

def _run(args, check:bool=False):
    'Run a scheduler command. `check` turns its complaint, or its absence, into an exception.'
    try: r = subprocess.run(args, capture_output=True)
    except FileNotFoundError:
        if check: raise RuntimeError(f'{args[0]} is not on this machine')
        return None
    r.stdout, r.stderr = _text(r.stdout), _text(r.stderr)
    if check and r.returncode: raise RuntimeError(f'{args[0]}: {(r.stderr or r.stdout).strip()}')
    return r

In [ ]:
#| export
def plist(script,                # the launcher to run
          tag:str=BEAT,          # the launchd label, and the plist filename
          every:int=EVERY        # seconds between beats
) -> bytes:
    'The job description launchd reads: run `script` every `every` seconds.'
    return plistlib.dumps(dict(Label=str(tag), ProgramArguments=[str(script)],
                               StartInterval=int(every), RunAtLoad=False))

class LaunchdBeat(Beat):
    'A LaunchAgent. What macOS wants, and it survives a sleeping laptop where cron does not.'
    def __init__(self,
                 dirn=None):     # where agents live; `~/Library/LaunchAgents` by default
        self.dirn = Path(ifnone(dirn, Path.home()/'Library/LaunchAgents')).expanduser()

    def _p(self, tag): return self.dirn/f'{tag}.plist'

    def install(self, script, tag:str=BEAT):
        self.dirn.mkdir(parents=True, exist_ok=True)
        self.uninstall(tag)
        self._p(tag).write_bytes(plist(script, tag))
        _run(['launchctl', 'bootstrap', f'gui/{os.getuid()}', str(self._p(tag))], check=True)
        return script

    def installed(self, tag:str=BEAT) -> str:
        p = self._p(tag)
        return plistlib.loads(p.read_bytes())['ProgramArguments'][0] if p.exists() else None

    def uninstall(self, tag:str=BEAT) -> bool:
        if not (p := self._p(tag)).exists(): return False
        # An agent that was written but never loaded has nothing to boot out, which is not an error
        _run(['launchctl', 'bootout', f'gui/{os.getuid()}/{tag}'])
        p.unlink()
        return True

launchd refuses a plist it cannot parse, and ignores keys it does not know, so the two
things worth asserting are that it parses and that the keys are spelled the way launchd spells
them.

In [ ]:
got = plistlib.loads(plist('/home/k/.pobblebonk/pobblebonk.sh'))
test_eq(got['Label'], 'pobblebonk')
test_eq(got['ProgramArguments'], ['/home/k/.pobblebonk/pobblebonk.sh'])
test_eq(got['StartInterval'], 60)

Reading an agent back and removing it are plain file operations, so they run anywhere. Only
the `launchctl` call in `install` needs a Mac, and it is the one line here that this notebook
cannot check.

In [ ]:
agents = d/'LaunchAgents'
agents.mkdir()
(agents/'pobblebonk.plist').write_bytes(plist('/home/k/.pobblebonk/pobblebonk.sh'))

launchd = LaunchdBeat(agents)
test_eq(launchd.installed(), '/home/k/.pobblebonk/pobblebonk.sh')
test_eq(launchd.uninstall(), True)
test_eq(launchd.installed(), None)
test_eq(launchd.uninstall(), False)

## Task Scheduler

Windows has no crontab and no plist. `schtasks` is a command, and the job lives somewhere you
never edit by hand, so there is no shared file to be careful with: `/f` overwrites a task of the
same name, which is exactly the replace-not-append rule. `/sc minute /mo 1` is its every-minute.

The path goes in bare. Quoting it here would be quoting it twice, since `subprocess` already
wraps an argument containing a space, and a Windows username with a space in it is not rare.

Reading a task back goes through `/xml`, because the human-readable `/query` output is translated
into the machine's display language and the field names move. That XML arrives UTF-16 encoded, in
a namespace, and with the command quoted whether or not it needed to be.

In [ ]:
#| export
def task_args(script,            # the launcher to run
              tag:str=BEAT,      # the task name
              every:int=EVERY    # seconds between beats, rounded down to whole minutes
) -> list:
    'The `schtasks` command that creates the task. `/f` is what makes a second install replace the first.'
    return ['schtasks', '/create', '/f', '/tn', str(tag), '/sc', 'minute',
            '/mo', str(max(1, int(every)//60)), '/tr', str(script)]

def _task_command(xml:str) -> str:
    'The program a `schtasks /xml` dump runs.'
    el = ET.fromstring(xml.strip()).find('.//{*}Command')
    return el.text.strip('"') if el is not None and el.text else None

class TaskBeat(Beat):
    'A Windows Task Scheduler entry, through `schtasks`.'
    def install(self, script, tag:str=BEAT):
        _run(task_args(script, tag), check=True)
        return script

    def installed(self, tag:str=BEAT) -> str:
        r = _run(['schtasks', '/query', '/tn', str(tag), '/xml', 'ONE'])
        return None if r is None or r.returncode else _task_command(r.stdout)

    def uninstall(self, tag:str=BEAT) -> bool:
        r = _run(['schtasks', '/delete', '/f', '/tn', str(tag)])
        return bool(r) and not r.returncode

No Windows here, so what this notebook can honestly check is the command itself. `/f` is the
one flag whose absence would break the replace rule quietly, by leaving the old task in place and
reporting success.

In [ ]:
args = task_args('C:\\Users\\k\\.pobblebonk\\pobblebonk.cmd', every=60)
assert '/f' in args
test_eq(args[args.index('/tn')+1], 'pobblebonk')
test_eq(args[args.index('/mo')+1], '1')
test_eq(args[-1], 'C:\\Users\\k\\.pobblebonk\\pobblebonk.cmd')

In [ ]:
# What `schtasks /query /xml ONE` gives back: UTF-16, namespaced, and the command quoted whether
# or not it needed to be.
xml = '''<?xml version="1.0" encoding="UTF-16"?>
<Task version="1.2" xmlns="http://schemas.microsoft.com/windows/2004/02/mit/task">
  <Triggers><TimeTrigger><Repetition><Interval>PT1M</Interval></Repetition></TimeTrigger></Triggers>
  <Actions Context="Author">
    <Exec><Command>"C:\\Users\\k\\.pobblebonk\\pobblebonk.cmd"</Command></Exec>
  </Actions>
</Task>'''

test_eq(_task_command(_text(xml.encode('utf-16'))), 'C:\\Users\\k\\.pobblebonk\\pobblebonk.cmd')

## Installing one

`beat` reads the platform. `install` is the whole module in four lines: write the launcher, hand
it to whatever this machine schedules with, give back the path so you know where to look when it
goes quiet.

`on` takes a `Beat` of your own. It is there so you can install a Windows task from a Windows
box and still test the wiring anywhere.

In [ ]:
#| export
def beat(platform:str=None) -> Beat:
    'The scheduler this machine actually has.'
    p = ifnone(platform, sys.platform)
    return LaunchdBeat() if p == 'darwin' else TaskBeat() if p == 'win32' else CronBeat()

def install(cmd:str,             # the command one beat runs
            dirn=None,           # where the launcher goes; `~/.pobblebonk` by default
            tag:str=BEAT,        # names the launcher, its log, and the scheduled job
            on:Beat=None         # a scheduler of your own; this machine's by default
) -> Path:
    'Write the launcher for `cmd` and have this machine run it every minute.'
    p = launcher(cmd, dirn, tag)
    ifnone(on, beat()).install(p, tag)
    return p

def installed(tag:str=BEAT, on:Beat=None) -> str:
    'The launcher `tag` runs now, or None.'
    return ifnone(on, beat()).installed(tag)

def uninstall(tag:str=BEAT, on:Beat=None) -> bool:
    'Stop the beat. False if there was nothing scheduled.'
    return ifnone(on, beat()).uninstall(tag)

In [ ]:
test_eq(type(beat('linux')), CronBeat)
test_eq(type(beat('darwin')), LaunchdBeat)
test_eq(type(beat('win32')), TaskBeat)

`install` has no logic of its own beyond that wiring, and wiring is exactly what breaks
silently: a launcher written to one path and a different one handed to the scheduler installs
cleanly and never ticks. A `Beat` that only remembers what it was given is enough to see it.

In [ ]:
class FakeBeat(Beat):
    'A scheduler that only remembers. Not exported: it exists to watch the wiring.'
    def __init__(self): self.jobs = {}
    def install(self, script, tag=BEAT): self.jobs[str(tag)] = str(script); return script
    def installed(self, tag=BEAT): return self.jobs.get(str(tag))
    def uninstall(self, tag=BEAT): return self.jobs.pop(str(tag), None) is not None

In [ ]:
fake = FakeBeat()
p = install('/bin/echo hi', d, tag='wired', on=fake)

# the file that exists on disk is the file the scheduler was given
test_eq(fake.installed('wired'), str(p))
assert p.exists()
test_eq(installed('wired', on=fake), str(p))

In [ ]:
test_eq(uninstall('wired', on=fake), True)
test_eq(installed('wired', on=fake), None)
test_eq(uninstall('wired', on=fake), False)

## What this does not do

It does not run anything itself, and it does not know what a schedule is. It installs one entry
that runs one file once a minute. The file runs your tick, the tick asks honker what is due, and
everything about *when* stays in the database where `pause`, `retries` and `catchup` can reach
it.

Two ticks can still overlap, when a callback takes longer than a minute. Locking against that
belongs in the tick, not here: `flock` in a crontab line is not available on Windows, and the
lock a heartbeat needs is a lock on the database, which the database can hold.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()